# Supplementary figures S1-S8

**Public-release note.** This notebook is part of a GitHub-ready version of the KMC memristor project. Notebook outputs were stripped to keep the repository lightweight, and path settings were adjusted to use repository-relative locations where needed.

**Purpose:** CSV-first notebook for rebuilding selected supplementary figures directly from exported tables.

**Main manuscript role:** Used to regenerate supplementary figures without rerunning the full simulation workflow.

**Default assumption:** run the notebook from inside this repository so that the `results/` directory can be discovered automatically.


# KMC supplementary figures S1–S8 — standalone, CSV-first

这个 notebook 只依赖你已经导出的 CSV；找不到时尽量用原始 raw CSV 重建，**不强依赖前面单元已经跑过**。  
目标是：**一键在页面里显示 S1–S8**，并同时保存到输出目录。

优先读取这些文件（与 notebook 同目录或 `BASE_DIR`）：

- `static_oat_param_impact.csv`
- `dynamic_param_impact.csv`
- `static_sens_case_metrics.csv`
- `dynamic_sens_case_metrics.csv`
- `static_sens_e50_tables.csv`
- `static_sens_points_raw.csv`
- `dynamic_sens_points_raw.csv`
- `S1_active_zone_e50.csv`
- `S1_active_zone_summary.csv`
- `S1_active_zone_topology.csv`

如果某一张图的“理想输入”不存在，就自动切到**当前能重建出的替代版本**。

In [ ]:
from pathlib import Path
import os, re, ast, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

warnings.filterwarnings("ignore")


def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for path in [start] + list(start.parents):
        if (path / "README.md").exists() and (path / "notebooks").exists() and (path / "results").exists():
            return path
    return start

# ==============================
# User-editable settings
# ==============================
REPO_ROOT = find_repo_root()
BASE_DIR = REPO_ROOT / "results" / "sensitivity"
OUT_DIR  = REPO_ROOT / "results" / "figures" / "supplementary"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SHOW_INLINE = True
SAVE_PNG = True
SAVE_PDF = True

# The notebook directory is also searched as a fallback
HERE = Path(".").resolve()

plt.rcParams["font.family"] = "DejaVu Sans"


In [ ]:

# ==============================
# 读取可用 CSV
# ==============================
static_imp   = read_csv_opt("static_oat_param_impact.csv")
dynamic_imp  = read_csv_opt("dynamic_param_impact.csv")
static_case  = read_csv_opt("static_sens_case_metrics.csv")
dynamic_case = read_csv_opt("dynamic_sens_case_metrics.csv")
static_e50   = read_csv_opt("static_sens_e50_tables.csv")
static_raw   = read_csv_opt("static_sens_points_raw.csv")
dynamic_raw  = read_csv_opt("dynamic_sens_points_raw.csv")

s1_e50   = read_csv_opt("S1_active_zone_e50.csv")
s1_sum   = read_csv_opt("S1_active_zone_summary.csv")
s1_topo  = read_csv_opt("S1_active_zone_topology.csv")

# 一些统一修补
for df in [static_raw, dynamic_raw, s1_sum, s1_topo]:
    if df is None:
        continue
    # sigma_E
    if "sigma_E" not in df.columns:
        alt = choose_col(df, ["sigma_final", "barrier_std_z", "barrier_std_x"], default=None)
        if alt is not None:
            df["sigma_E"] = pd.to_numeric(df[alt], errors="coerce")
    # lc
    if "lc" not in df.columns:
        alt = choose_col(df, ["lc_final"], default=None)
        if alt is not None:
            df["lc"] = pd.to_numeric(df[alt], errors="coerce")
    # Delta
    if "Delta" not in df.columns:
        alt = choose_col(df, ["Delta_final"], default=None)
        if alt is not None:
            df["Delta"] = pd.to_numeric(df[alt], errors="coerce")
        elif "sigma_E" in df.columns and "T" in df.columns:
            df["Delta"] = pd.to_numeric(df["sigma_E"], errors="coerce") / (KB_EV * pd.to_numeric(df["T"], errors="coerce"))

# standardize numeric for raw points
if static_raw is not None:
    static_raw = ensure_numeric(static_raw, ["m_target","m_actual","E","T","formed","t_set","sigma_E","lc","Delta",
                                             "branches","branches_crit","tortuosity_nm","tortuosity_nm_crit",
                                             "neck_nm","neck_nm_crit","E_over_E50_target"])
if dynamic_raw is not None:
    dynamic_raw = ensure_numeric(dynamic_raw, ["m_target","m_actual","E","T","formed","t_set","sigma_E","lc","Delta",
                                               "sigma_final","lc_final","Delta_final","m_final","formed","branches",
                                               "branches_crit","tortuosity_nm","tortuosity_nm_crit","E_over_E50_target",
                                               "dynamic_update_events","dynamic_alpha_sigma"])

print("Loaded tables:",
      {k:v.shape if isinstance(v,pd.DataFrame) else None for k,v in {
          "static_imp":static_imp, "dynamic_imp":dynamic_imp, "static_case":static_case, "dynamic_case":dynamic_case,
          "static_e50":static_e50, "static_raw":static_raw, "dynamic_raw":dynamic_raw,
          "s1_e50":s1_e50, "s1_sum":s1_sum, "s1_topo":s1_topo
      }.items()})


In [ ]:

# ==============================
# 从现有 CSV 重建一些中间表
# ==============================
def build_scaling_df_from_static_raw(static_raw, static_e50):
    if static_raw is None or static_e50 is None:
        return None
    need_cols = {"param","level_name","m_target","E"}
    if not need_cols.issubset(static_raw.columns) or not {"param","level_name","m_target","E50","wE"}.issubset(static_e50.columns):
        return None
    df = static_raw.copy()
    map_e50 = static_e50[["param","level_name","m_target","E50","wE"]].copy()
    out = df.merge(map_e50, on=["param","level_name","m_target"], how="left")
    out["E_norm"] = out["E"] / out["E50"]
    # 统一 topology 列
    if "branches_crit" in out.columns:
        out["branches_topo"] = out["branches_crit"]
    elif "branches" in out.columns:
        out["branches_topo"] = out["branches"]
    if "tortuosity_nm_crit" in out.columns:
        out["tortuosity_topo"] = out["tortuosity_nm_crit"]
    elif "tortuosity_nm" in out.columns:
        out["tortuosity_topo"] = out["tortuosity_nm"]
    if "lc" not in out.columns and "lc_final" in out.columns:
        out["lc"] = out["lc_final"]
    if "Delta" not in out.columns and "Delta_final" in out.columns:
        out["Delta"] = out["Delta_final"]
    return out

scaling_df = build_scaling_df_from_static_raw(static_raw, static_e50)

def quadratic_fit_stats(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if len(x) < 5:
        return np.nan, np.nan, None
    coef = np.polyfit(x, y, 2)
    yhat = coef[0]*x*x + coef[1]*x + coef[2]
    ss_res = np.sum((y-yhat)**2)
    ss_tot = np.sum((y-np.mean(y))**2)
    r2 = np.nan if ss_tot <= 0 else 1 - ss_res/ss_tot
    rmse = np.sqrt(np.mean((y-yhat)**2))
    return rmse, r2, coef

def alpha_scan(df, ycol, alpha_grid=np.linspace(-1.0, 2.0, 61), xi=False):
    if df is None or ycol not in df.columns:
        return pd.DataFrame(columns=["alpha","r2","rmse"])
    g = df.copy()
    g = g[np.isfinite(g["E_norm"]) & np.isfinite(g["Delta"]) & np.isfinite(g["lc"]) & np.isfinite(g[ycol])]
    if len(g) < 8:
        return pd.DataFrame(columns=["alpha","r2","rmse"])
    lc_ref = np.nanmedian(g["lc"].values)
    out = []
    for a in alpha_grid:
        if xi:
            x = (g["E_norm"] - 1.0) / (g["Delta"] * ((g["lc"]/lc_ref) ** a))
        else:
            x = g["E_norm"] / (g["Delta"] * ((g["lc"]/lc_ref) ** a))
        rmse, r2, coef = quadratic_fit_stats(x.values, g[ycol].values)
        out.append({"alpha":a, "r2":r2, "rmse":rmse})
    return pd.DataFrame(out)

score_psi_br = alpha_scan(scaling_df, "branches_topo", xi=False) if scaling_df is not None else None
score_psi_to = alpha_scan(scaling_df, "tortuosity_topo", xi=False) if scaling_df is not None else None
score_xi_br  = alpha_scan(scaling_df, "branches_topo", xi=True) if scaling_df is not None else None
score_xi_to  = alpha_scan(scaling_df, "tortuosity_topo", xi=True) if scaling_df is not None else None

print("scaling_df:", None if scaling_df is None else scaling_df.shape)


In [ ]:

# ==============================
# Figure S1
# ==============================
def make_S1():
    # 优先用专门的 S1 CSV；没有就跳过
    if s1_e50 is None or s1_topo is None:
        print("[skip] S1: missing S1_active_zone_* csv")
        return

    records = []
    for (zmax, m), grp in s1_topo.groupby(["zmax","m_target"]):
        row = s1_e50[(s1_e50["zmax"] == zmax) & (s1_e50["m_target"] == m)]
        if row.empty:
            continue
        e50 = float(row["E50"].iloc[0]) if "E50" in row.columns else np.nan
        we  = float(row["wE"].iloc[0])  if "wE"  in row.columns else np.nan
        branches = nearest_valid_in_group(grp, "E", "branches", e50)
        tort     = nearest_valid_in_group(grp, "E", "tortuosity", e50)
        records.append({"zmax":zmax, "m_target":m, "E50":e50, "wE":we, "branches":branches, "tortuosity":tort})
    plot_df = pd.DataFrame(records).sort_values(["m_target","zmax"])
    if len(plot_df) == 0:
        print("[skip] S1: no valid rows")
        return

    z_dense = np.arange(int(plot_df["zmax"].min()), int(plot_df["zmax"].max())+1)

    fig = plt.figure(figsize=(16,10))
    gs = fig.add_gridspec(2,2, hspace=0.34, wspace=0.24)
    panels = [
        ("E50", r"$E_{50}$ (V/nm)", "(a) Estimated $E_{50}$ vs active-zone depth"),
        ("wE",  r"$w_E$ (V/nm)", "(b) Transition width vs active-zone depth"),
        ("branches", "Branches", "(c) Branches near estimated $E_{50}$"),
        ("tortuosity", "Tortuosity", "(d) Tortuosity near estimated $E_{50}$"),
    ]
    for i,(col,ylab,title) in enumerate(panels):
        ax = fig.add_subplot(gs[i//2, i%2])
        for m, g in plot_df.groupby("m_target"):
            g = g.sort_values("zmax")
            yd = norm_interp(g["zmax"], g[col], z_dense)
            ax.plot(z_dense, yd, linewidth=2.0, label=f"m={m:.2f}")
            ax.scatter(g["zmax"], g[col], s=35)
        ax.set_title(title, loc="left")
        ax.set_xlabel(r"Active-zone definition ($z \leq z_{\max}$)")
        ax.set_ylabel(ylab)
        ax.grid(alpha=0.35)
        ax.legend()
    fig.suptitle("Figure S1. Active-zone depth definition comparison", fontsize=26, weight="bold", y=0.98)
    fig.tight_layout(rect=[0,0,1,0.95])
    savefig_all(fig, "Figure_S1_allinone")


In [ ]:

# ==============================
# Figure S2
# 更多 barrier heatmaps
# ==============================
def make_S2():
    if dynamic_raw is None or "em_z" not in dynamic_raw.columns:
        print("[skip] S2: dynamic_sens_points_raw.csv or em_z missing")
        return

    df = dynamic_raw.copy()
    # baseline static 优先；没有就 baseline dynamic
    if {"param","level_name","label"}.issubset(df.columns):
        sub = df[(df["param"]=="BASELINE") & (df["level_name"]=="base")].copy()
    else:
        sub = df.copy()
    if "mode" in sub.columns and (sub["mode"]=="static").any():
        sub = sub[sub["mode"]=="static"].copy()

    wanted = ["m025_sub","m025_thr","m025_sup","m090_sub","m090_thr","m090_sup"]
    rows = []
    for lab in wanted:
        g = sub[sub["label"]==lab]
        if len(g):
            rows.append(g.iloc[0])
    if len(rows) == 0:
        # fallback 取前 6 个
        rows = [r for _,r in sub.head(6).iterrows()]
    mats = []
    for r in rows:
        arr = parse_array_cell(r["em_z"])
        mat = reshape_heatmap(arr, prefer=(24,36))
        mats.append(mat)
    finite = [m[np.isfinite(m)] for m in mats if m.size]
    if len(finite) == 0:
        print("[skip] S2: no valid em_z arrays")
        return
    vmin = min(np.nanmin(x) for x in finite)
    vmax = max(np.nanmax(x) for x in finite)

    n = len(rows)
    ncol = 3
    nrow = int(np.ceil(n/ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(15, 4.5*nrow))
    axes = np.atleast_1d(axes).ravel()

    im = None
    for ax, r, mat in zip(axes, rows, mats):
        im = ax.imshow(mat.T, origin="lower", aspect="auto", vmin=vmin, vmax=vmax)
        ttl = pretty_label(r.get("label","pt"))
        if "m_target" in r.index and "E" in r.index:
            ttl += f"\nm={float(r['m_target']):.2f}, E={float(r['E']):.3f}"
        ax.set_title(ttl)
        ax.set_xlabel("x cell")
        ax.set_ylabel("z cell")
    for ax in axes[len(rows):]:
        ax.axis("off")
    cbar = fig.colorbar(im, ax=axes.tolist(), fraction=0.025, pad=0.02)
    cbar.set_label(r"Vertical migration barrier $E_m^z$ (eV)")
    fig.suptitle("Figure S2. Additional barrier heatmaps", fontsize=26, weight="bold", y=0.98)
    fig.tight_layout(rect=[0,0,1,0.95])
    savefig_all(fig, "Figure_S2_allinone")


In [ ]:

# ==============================
# Figure S3
# 优先：真正的 mT fixed-E 结果
# fallback：当前已有代表点的 fixed-drive 结果
# ==============================
def make_S3():
    # fallback 用当前现有 raw/case 数据，不强行依赖 phase2 扫描文件
    if static_raw is None and dynamic_raw is None:
        print("[skip] S3: no raw csv")
        return

    # 优先用 dynamic_raw baseline 六点
    if dynamic_raw is not None and {"param","level_name","label","E_over_E50_target"}.issubset(dynamic_raw.columns):
        df = dynamic_raw[(dynamic_raw["param"]=="BASELINE") & (dynamic_raw["level_name"]=="base")].copy()
        if len(df):
            # 每个 label & mode 聚合
            agg = []
            grpcols = ["label","mode","m_target","E_over_E50_target"]
            for keys, g in df.groupby(grpcols):
                rec = {k:v for k,v in zip(grpcols, keys)}
                rec["formed_prob"] = np.nanmean(g["formed"]) if "formed" in g.columns else np.nan
                if "t_set" in g.columns:
                    rec["log10_t_set"] = np.nanmedian(np.log10(g["t_set"].where(g["t_set"]>0)))
                else:
                    rec["log10_t_set"] = np.nan
                rec["branches"] = np.nanmedian(g["branches_crit"]) if "branches_crit" in g.columns else np.nan
                rec["tortuosity"] = np.nanmedian(g["tortuosity_nm_crit"]) if "tortuosity_nm_crit" in g.columns else np.nan
                agg.append(rec)
            dd = pd.DataFrame(agg)
            fig, axes = plt.subplots(2,2, figsize=(14,10))
            axes = axes.ravel()
            panels = [
                ("formed_prob","Formation probability"),
                ("log10_t_set",r"$\log_{10}(t_{set})$"),
                ("branches","Critical branches"),
                ("tortuosity","Critical tortuosity"),
            ]
            for ax,(col,ttl) in zip(axes, panels):
                for (m, mode), g in dd.groupby(["m_target","mode"]):
                    g = g.sort_values("E_over_E50_target")
                    ax.plot(g["E_over_E50_target"], g[col], marker="o", linewidth=2, label=f"m={m:.2f}, {mode}")
                ax.set_title(ttl, loc="left")
                ax.set_xlabel(r"Representative drive ($E/E_{50}$ target)")
                ax.set_ylabel("")
                ax.grid(alpha=0.35)
            axes[0].legend(ncol=2, fontsize=9)
            fig.suptitle("Figure S3. Additional fixed-drive representative results", fontsize=26, weight="bold", y=0.98)
            fig.tight_layout(rect=[0,0,1,0.95])
            savefig_all(fig, "Figure_S3_allinone")
            return

    print("[skip] S3: no usable representative raw points")


In [ ]:

# ==============================
# Figure S4
# tortuosity scaling suite
# ==============================
def make_S4():
    if scaling_df is None or "tortuosity_topo" not in scaling_df.columns:
        print("[skip] S4: scaling_df missing")
        return
    df = scaling_df.copy()
    df = df[np.isfinite(df["E_norm"]) & np.isfinite(df["Delta"]) & np.isfinite(df["lc"]) & np.isfinite(df["tortuosity_topo"])]
    if len(df) < 8:
        print("[skip] S4: too few valid rows")
        return

    fig, axes = plt.subplots(2,2, figsize=(14,10))
    ax1, ax2, ax3, ax4 = axes.ravel()

    sc = ax1.scatter(df["E_norm"], df["tortuosity_topo"], c=df["Delta"], s=60 + 80*safe_div(df["lc"], np.nanmedian(df["lc"])),
                     alpha=0.8)
    ax1.set_title("(a) Tortuosity vs normalized drive", loc="left")
    ax1.set_xlabel(r"$E/E_{50}$")
    ax1.set_ylabel("Tortuosity")
    fig.colorbar(sc, ax=ax1, fraction=0.046, pad=0.03, label=r"$\Delta$")

    sc2 = ax2.scatter(df["Delta"], df["E_norm"], c=df["tortuosity_topo"], s=60 + 80*safe_div(df["lc"], np.nanmedian(df["lc"])),
                      alpha=0.8)
    ax2.set_title("(b) Tortuosity in $(\Delta, E/E_{50})$ space", loc="left")
    ax2.set_xlabel(r"$\Delta$")
    ax2.set_ylabel(r"$E/E_{50}$")
    fig.colorbar(sc2, ax=ax2, fraction=0.046, pad=0.03, label="Tortuosity")

    if score_psi_to is not None and len(score_psi_to):
        t = score_psi_to.sort_values("alpha")
        ax3.plot(t["alpha"], t["r2"], marker="o", linewidth=2)
        best = t.sort_values("r2", ascending=False).iloc[0]
        ax3.axvline(best["alpha"], ls="--", lw=1.0, color="gray")
        ax3.text(best["alpha"], best["r2"], f"best α={best['alpha']:.2f}", fontsize=10, ha="left", va="bottom")
    ax3.set_title("(c) Alpha scan for tortuosity", loc="left")
    ax3.set_xlabel(r"$\alpha$")
    ax3.set_ylabel(r"$R^2$")

    # best Psi collapse
    if score_psi_to is not None and len(score_psi_to):
        best = score_psi_to.sort_values("r2", ascending=False).iloc[0]
        a = float(best["alpha"])
        lc_ref = np.nanmedian(df["lc"].values)
        df["Psi_best"] = df["E_norm"] / (df["Delta"] * ((df["lc"]/lc_ref) ** a))
        rmse, r2, coef = quadratic_fit_stats(df["Psi_best"].values, df["tortuosity_topo"].values)
        ax4.scatter(df["Psi_best"], df["tortuosity_topo"], c=df["m_target"], alpha=0.85)
        if coef is not None:
            xx = np.linspace(np.nanmin(df["Psi_best"]), np.nanmax(df["Psi_best"]), 250)
            yy = coef[0]*xx*xx + coef[1]*xx + coef[2]
            ax4.plot(xx, yy, color="black", lw=2, label=f"quad fit, R²={r2:.3f}")
            ax4.legend()
    ax4.set_title("(d) Best Psi collapse for tortuosity", loc="left")
    ax4.set_xlabel(r"$\Psi_\alpha$")
    ax4.set_ylabel("Tortuosity")

    fig.suptitle("Figure S4. Tortuosity scaling suite", fontsize=26, weight="bold", y=0.98)
    fig.tight_layout(rect=[0,0,1,0.95])
    savefig_all(fig, "Figure_S4_allinone")


In [ ]:

# ==============================
# Figure S5
# alpha 扫描与 Psi/Xi 对比
# ==============================
def make_S5():
    if score_psi_br is None or score_psi_to is None or score_xi_br is None or score_xi_to is None:
        print("[skip] S5: alpha scans missing")
        return
    fig, axes = plt.subplots(1,2, figsize=(14,5.4))
    ax1, ax2 = axes

    if len(score_psi_br):
        t = score_psi_br.sort_values("alpha")
        ax1.plot(t["alpha"], t["r2"], marker="o", linewidth=2, label="Psi → branches")
    if len(score_xi_br):
        t = score_xi_br.sort_values("alpha")
        ax1.plot(t["alpha"], t["r2"], marker="s", linewidth=2, label="Xi → branches")
    ax1.set_title("(a) Alpha scan for branch descriptor", loc="left")
    ax1.set_xlabel(r"$\alpha$")
    ax1.set_ylabel(r"$R^2$")
    ax1.legend()

    if len(score_psi_to):
        t = score_psi_to.sort_values("alpha")
        ax2.plot(t["alpha"], t["r2"], marker="o", linewidth=2, label="Psi → tortuosity")
    if len(score_xi_to):
        t = score_xi_to.sort_values("alpha")
        ax2.plot(t["alpha"], t["r2"], marker="s", linewidth=2, label="Xi → tortuosity")
    ax2.set_title("(b) Alpha scan for tortuosity descriptor", loc="left")
    ax2.set_xlabel(r"$\alpha$")
    ax2.set_ylabel(r"$R^2$")
    ax2.legend()

    fig.suptitle("Figure S5. Alpha scan and Psi/Xi comparison", fontsize=26, weight="bold", y=0.98)
    fig.tight_layout(rect=[0,0,1,0.93])
    savefig_all(fig, "Figure_S5_allinone")


In [ ]:

# ==============================
# Figure S6
# static OAT 原始轨迹图
# ==============================
def make_S6():
    if static_case is None:
        print("[skip] S6: static_sens_case_metrics.csv missing")
        return
    df = static_case.copy()
    if "param" not in df.columns or "level_name" not in df.columns:
        print("[skip] S6: param/level_name missing")
        return

    metrics = [
        ("formed_prob_mean6", "Formation probability"),
        ("branches_crit_med_mean6", "Critical branches"),
        ("E50_m025", r"$E_{50}$ at m=0.25"),
        ("E50_m090", r"$E_{50}$ at m=0.90"),
    ]
    fig, axes = plt.subplots(2,2, figsize=(14,10))
    axes = axes.ravel()

    for ax, (col, ttl) in zip(axes, metrics):
        if col not in df.columns:
            ax.text(0.5, 0.5, f"Missing:\n{col}", ha="center", va="center", transform=ax.transAxes)
            ax.axis("off")
            continue
        for p, g in df.groupby("param"):
            g = g.copy()
            # 让 base/lvl0/lvl1/lvl2 有顺序
            order_map = {"base":-1, "lvl0":0, "lvl1":1, "lvl2":2}
            g["_ord"] = g["level_name"].map(order_map).fillna(999)
            g = g.sort_values("_ord")
            ax.plot(g["level_name"], g[col], marker="o", linewidth=1.8, label=p)
        ax.set_title(ttl, loc="left")
        ax.set_xlabel("Level")
        ax.set_ylabel("")
        ax.grid(alpha=0.35)
    axes[0].legend(ncol=2, fontsize=8)
    fig.suptitle("Figure S6. Static OAT raw trajectories", fontsize=26, weight="bold", y=0.98)
    fig.tight_layout(rect=[0,0,1,0.95])
    savefig_all(fig, "Figure_S6_allinone")


In [ ]:

# ==============================
# Figure S7
# dynamic 参数敏感性
# ==============================
def make_S7():
    if dynamic_imp is None and dynamic_case is None:
        print("[skip] S7: dynamic csv missing")
        return

    metric_cols = [
        "abs_delta_formed_prob_mean6",
        "abs_delta_log10_t_set_median_formed_mean6",
        "abs_delta_branches_crit_median_formed_mean6",
        "delta_m_final_mean_mean6",
        "delta_Delta_final_mean_mean6",
    ]

    if dynamic_imp is not None:
        df = dynamic_imp.copy()
    else:
        # 从 dynamic_case 粗略重建 impact（只取各参数 across levels 的 range）
        rows = []
        for p, g in dynamic_case.groupby("param"):
            rec = {"param":p}
            for c in metric_cols:
                if c in g.columns:
                    base = np.nanmedian(np.abs(g[c].values))
                    ran = np.nanmax(g[c].values) - np.nanmin(g[c].values)
                    rec[c] = ran if not np.isfinite(base) or abs(base)<1e-12 else ran/max(abs(base),1e-12)
            rows.append(rec)
        df = pd.DataFrame(rows)

    use_cols = [c for c in metric_cols if c in df.columns]
    if len(use_cols) == 0:
        print("[skip] S7: no usable dynamic metric columns")
        return

    # 去掉全零行
    df = df.copy()
    df["row_sum"] = df[use_cols].fillna(0).abs().sum(axis=1)
    df = df[df["row_sum"] > 0].copy()
    if len(df) == 0:
        print("[skip] S7: all-zero rows")
        return

    fig, axes = plt.subplots(1,2, figsize=(14.8,5.7))
    ax1, ax2 = axes

    hm = df.set_index("param")[use_cols]
    im = ax1.imshow(hm.to_numpy(dtype=float), aspect="auto")
    ax1.set_yticks(range(len(hm.index)))
    ax1.set_yticklabels(hm.index)
    ax1.set_xticks(range(len(use_cols)))
    ax1.set_xticklabels([
        r"|ΔP$_{form}$|",
        r"|Δlog$_{10}$(t$_{set}$)|",
        r"|Δbranches$_{crit}$|",
        r"Δm$_{final}$",
        r"ΔΔ$_{final}$",
    ][:len(use_cols)], rotation=18, ha="right")
    ax1.set_title("(a) Dynamic-parameter heatmap", loc="left")
    for i in range(hm.shape[0]):
        for j in range(hm.shape[1]):
            ax1.text(j, i, f"{hm.iloc[i,j]:.2f}", ha="center", va="center", fontsize=10)
    fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.03, label="Impact")

    if "abs_delta_branches_crit_median_formed_mean6" in df.columns:
        rank = df[["param","abs_delta_branches_crit_median_formed_mean6"]].sort_values("abs_delta_branches_crit_median_formed_mean6", ascending=True)
        ax2.barh(rank["param"], rank["abs_delta_branches_crit_median_formed_mean6"])
        for y, v in zip(rank["param"], rank["abs_delta_branches_crit_median_formed_mean6"]):
            ax2.text(v + 0.01, y, f"{v:.2f}", va="center", fontsize=10)
        ax2.set_xlabel(r"Relative impact on $|\Delta branches_{crit}|$")
    else:
        # fallback 任意第一列
        c = use_cols[0]
        rank = df[["param",c]].sort_values(c, ascending=True)
        ax2.barh(rank["param"], rank[c])
        ax2.set_xlabel(c)
    ax2.set_title("(b) Dynamic-parameter ranking", loc="left")
    ax2.grid(axis="x", alpha=0.35)

    fig.suptitle("Figure S7. Dynamic parameter sensitivity", fontsize=26, weight="bold", y=0.98)
    fig.tight_layout(rect=[0,0,1,0.93])
    savefig_all(fig, "Figure_S7_allinone")


In [ ]:

# ==============================
# Figure S8
# representative-point static/dynamic traces
# ==============================
def make_S8():
    if dynamic_raw is None:
        print("[skip] S8: dynamic_sens_points_raw.csv missing")
        return
    df = dynamic_raw.copy()
    if not {"label","mode","param","level_name"}.issubset(df.columns):
        print("[skip] S8: required grouping cols missing")
        return

    # 只看 baseline base
    sub = df[(df["param"]=="BASELINE") & (df["level_name"]=="base")].copy()
    if len(sub) == 0:
        sub = df.copy()

    labels = ["m025_sub","m025_thr","m025_sup","m090_sub","m090_thr","m090_sup"]
    labels = [lab for lab in labels if (sub["label"] == lab).any()]
    if len(labels) == 0:
        labels = list(sub["label"].dropna().unique())[:6]

    def avg_trace(group, col):
        arrs = []
        for v in group[col].dropna():
            a = parse_array_cell(v).ravel()
            a = a[np.isfinite(a)]
            if len(a) >= 2:
                arrs.append(a)
        if len(arrs) == 0:
            return None, None
        grid = np.linspace(0, 1, 120)
        mats = []
        for a in arrs:
            xold = np.linspace(0, 1, len(a))
            mats.append(np.interp(grid, xold, a))
        mats = np.asarray(mats, dtype=float)
        return grid, np.nanmean(mats, axis=0)

    fig, axes = plt.subplots(2, 3, figsize=(15.8, 8.8))
    axes = axes.ravel()

    for ax, lab in zip(axes, labels):
        g = sub[sub["label"] == lab]
        gm_s, ym_s = avg_trace(g[g["mode"]=="static"], "trace_m")
        gm_d, ym_d = avg_trace(g[g["mode"]=="dynamic"], "trace_m")
        gs_s, ys_s = avg_trace(g[g["mode"]=="static"], "trace_sigma")
        gs_d, ys_d = avg_trace(g[g["mode"]=="dynamic"], "trace_sigma")

        if ym_s is not None:
            ax.plot(gm_s, ym_s, linewidth=2, label="m(t), static")
        if ym_d is not None:
            ax.plot(gm_d, ym_d, linewidth=2, label="m(t), dynamic")
        if ys_s is not None:
            ax.plot(gs_s, ys_s, linewidth=1.6, ls="--", label=r"$\sigma_E(t)$, static")
        if ys_d is not None:
            ax.plot(gs_d, ys_d, linewidth=1.6, ls="--", label=r"$\sigma_E(t)$, dynamic")
        ax.set_title(pretty_label(lab))
        ax.set_xlabel("Normalized progress")
        ax.set_ylabel("Trace value")
        ax.grid(alpha=0.35)

    for ax in axes[len(labels):]:
        ax.axis("off")

    handles, labels_ = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels_, ncol=4, loc="upper center", bbox_to_anchor=(0.5, 0.93))
    fig.suptitle("Figure S8. Representative-point static/dynamic traces", fontsize=26, weight="bold", y=0.98)
    fig.tight_layout(rect=[0,0,1,0.90])
    savefig_all(fig, "Figure_S8_allinone")


In [ ]:

# ==============================
# Figure 6（可选，一起补）
# ==============================
def make_Figure6_optional():
    if static_imp is None or dynamic_imp is None:
        print("[skip] Figure 6 optional: impact csv missing")
        return

    static_cols = [
        ("formed_prob_mean6", "Formation\nprobability"),
        ("log10_t_set_med_mean6", r"$\log_{10}(t_{set})$"),
        ("branches_crit_med_mean6", "Branches at\ncritical state"),
        ("tortuosity_crit_med_mean6", "Tortuosity at\ncritical state"),
        ("E50_m025", r"$E_{50}$ at m = 0.25"),
        ("R2_Psi_branches", r"$R^2(\Psi,\mathrm{branches})$"),
    ]
    dyn_cols = [
        ("abs_delta_formed_prob_mean6", r"$|\Delta P_{form}|$"),
        ("abs_delta_log10_t_set_median_formed_mean6", r"$|\Delta \log_{10}(t_{set})|$"),
        ("abs_delta_branches_crit_median_formed_mean6", r"$|\Delta branches_{crit}|$"),
        ("delta_m_final_mean_mean6", r"$\Delta m_{final}$"),
        ("delta_Delta_final_mean_mean6", r"$\Delta \Delta_{final}$"),
    ]

    si = static_imp.copy()
    di = dynamic_imp.copy()
    si = si[si[[c for c,_ in static_cols if c in si.columns]].fillna(0).abs().sum(axis=1) > 0]
    di = di[di[[c for c,_ in dyn_cols if c in di.columns]].fillna(0).abs().sum(axis=1) > 0]

    fig = plt.figure(figsize=(16,10))
    gs = fig.add_gridspec(2,2, hspace=0.34, wspace=0.28)

    ax1 = fig.add_subplot(gs[0,0])
    use_sc = [c for c,_ in static_cols if c in si.columns]
    im1 = ax1.imshow(si[use_sc].to_numpy(dtype=float), aspect="auto")
    ax1.set_xticks(range(len(use_sc)))
    ax1.set_xticklabels([lab for c,lab in static_cols if c in use_sc], rotation=18, ha="right")
    ax1.set_yticks(range(len(si)))
    ax1.set_yticklabels(si["param"] if "param" in si.columns else np.arange(len(si)))
    ax1.set_title("(a) Static sensitivity heatmap", loc="left")
    for i in range(len(si)):
        for j,c in enumerate(use_sc):
            ax1.text(j,i,f"{si.iloc[i][c]:.2f}",ha="center",va="center",fontsize=10)
    fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.03, label="Impact")

    ax2 = fig.add_subplot(gs[0,1])
    if "R2_Psi_branches" in si.columns:
        rk = si[["param","R2_Psi_branches"]].sort_values("R2_Psi_branches", ascending=True)
        ax2.barh(rk["param"], rk["R2_Psi_branches"])
        for y,v in zip(rk["param"], rk["R2_Psi_branches"]):
            ax2.text(v+0.01, y, f"{v:.2f}", va="center", fontsize=10)
        ax2.set_xlabel(r"Relative impact on $R^2(\Psi,\mathrm{branches})$")
    ax2.set_title(r"(b) Ranking by $R^2(\Psi,\mathrm{branches})$", loc="left")
    ax2.grid(axis="x", alpha=0.35)

    ax3 = fig.add_subplot(gs[1,0])
    use_dc = [c for c,_ in dyn_cols if c in di.columns]
    im2 = ax3.imshow(di[use_dc].to_numpy(dtype=float), aspect="auto")
    ax3.set_xticks(range(len(use_dc)))
    ax3.set_xticklabels([lab for c,lab in dyn_cols if c in use_dc], rotation=18, ha="right")
    ax3.set_yticks(range(len(di)))
    ax3.set_yticklabels(di["param"] if "param" in di.columns else np.arange(len(di)))
    ax3.set_title("(c) Dynamic sensitivity heatmap", loc="left")
    for i in range(len(di)):
        for j,c in enumerate(use_dc):
            ax3.text(j,i,f"{di.iloc[i][c]:.2f}",ha="center",va="center",fontsize=10)
    fig.colorbar(im2, ax=ax3, fraction=0.046, pad=0.03, label="Impact")

    ax4 = fig.add_subplot(gs[1,1])
    if dynamic_case is not None and {"branches_crit_median_formed_static_mean6","branches_crit_median_formed_dynamic_mean6"}.issubset(dynamic_case.columns):
        g = dynamic_case.copy()
        base = g[(g["param"]=="BASELINE") & (g["level_name"]=="base")]
        if len(base):
            s = float(base["branches_crit_median_formed_static_mean6"].iloc[0])
            d = float(base["branches_crit_median_formed_dynamic_mean6"].iloc[0])
            ax4.plot([0,1],[s,d], marker="o")
            ax4.set_xticks([0,1]); ax4.set_xticklabels(["Static","Dynamic"])
            ax4.set_ylabel("Branches at critical state")
    ax4.set_title("(d) Static vs dynamic robustness check", loc="left")
    ax4.grid(alpha=0.35)

    fig.suptitle("Figure 6. Sensitivity analysis and dynamic robustness check", fontsize=26, weight="bold", y=0.98)
    fig.tight_layout(rect=[0,0,1,0.95])
    savefig_all(fig, "Figure_6_optional")


In [ ]:

# ==============================
# 一键输出：S1–S8（以及可选 Figure 6）
# ==============================
make_S1()
make_S2()
make_S3()
make_S4()
make_S5()
make_S6()
make_S7()
make_S8()

# 如需要，取消下一行注释
# make_Figure6_optional()

print("\nSaved to:", OUT_DIR)
